In [2]:
from pynq import Overlay
from pynq.lib import MicroblazeLibrary
from IPython.display import clear_output
import time

# Load overlay and initialize UART
base = Overlay("base.bit")
lib = MicroblazeLibrary(base.iop_arduino, ['uart'])
uart = lib.uart_open(1, 0)  # Arduino: TXD=1 (unused), RXD=0 (connect to sensor TX)

def read_distance():
    """Read a complete distance packet from the sensor."""
    while True:
        # Wait for 'R'
        byte = [0x00]
        uart.read(byte, 1)
        if chr(byte[0]) != 'R':
            continue
            
        # Read 3 digits and CR
        data = []
        for _ in range(4):
            byte = [0x00]
            uart.read(byte, 1)
            data.append(byte[0])
            
        # Convert to string and check validity
        try:
            distance_str = ''.join(chr(b) for b in data[:3])
            if data[3] == 0x0d and distance_str.isdigit():
                return int(distance_str)
        except:
            pass

try:
    print("Starting sensor readings...")
    print("(Move objects in front of sensor to test)")
    
    while True:
        distance = read_distance()
        
        clear_output(wait=True)
        print("\nSensor Reading:")
        print("--------------")
        print(f"Distance: {distance} inches ({distance * 2.54:.1f} cm)")
        
        # Add interpretation
        if distance == 255:
            print("Status: No object detected/Maximum range")
        elif distance == 6:
            print("Status: Object at minimum range")
        else:
            print("Status: Valid intermediate reading")

except KeyboardInterrupt:
    print("\nExiting...")
finally:
    uart.close()
    print("UART connection closed")


Sensor Reading:
--------------
Distance: 19 inches (48.3 cm)
Status: Valid intermediate reading

Exiting...
UART connection closed
